**Problem 2**

You want to maintain the temperature of a room. You want to maintain the speed of a mechanism under varying loads. You want to maintain the heading of a robot on imperfect terrain. You want your drone to remain parallel with the ground to maintain balance.

All of these are control problems. You want to maintain a **stable state**, while natural imperfections fight you. 

Let's first consider the temperature problem and say you want to keep the room at a temperature greater than equal to $25 \degree \text{C}$. A simple way to do so is to turn on the AC if it's hotter than that, and turn it off if it's cooler. The controller output $u(t)$ at any point in time would be:
$$
u(t) = 
\begin{cases}
1 & \text{if} \; T \geq 25 \degree \text{C} \\
0 & \text{if} \; T < 25 \degree \text{C}
\end{cases}
$$

This is called a **bang-bang controller**. It's also known as other names, such as hysteresis, 2-step, or on-off controller. The idea is simple: only act if the state is not what you want. You have a reference point, called the **setpoint** ($SP$), and the actual sensor value, the **process valuable** ($PV$). We simply measure $PV$ relative to $SP$. Then, we can very easily implement it as such:

In [ ]:
class BangBang:
    def __init__(self, setpoint):
        self.setpoint = setpoint

    def control(self, current_value):
        if current_value < self.setpoint:
            return 1  # Turn on the actuator (e.g., heater)
        else:
            return 0  # Turn off the actuator (e.g., heater)

In this model, we simulate our AC unit to have a maximum cooling rate of $u_{max} \degree \text{C} /\text{s}$. 

We also model the heat exchange of the room as:
$$
\dfrac{dT}{dt} = -\dfrac{T + T_{ambient}}{\tau} + u + d
$$
where:
- $T(t)$ is the temperature of the room at time t
- $T_{ambient}$ is the ambient temperature
- $\tau$ is the thermal time constant of the system
- $u$ is the control input
- $d \sim \mathcal{N}(0, \sigma^2)$ is random disturbance noise, sampled from a Gaussian distribution

Discretising it, we yield the equation:
$$
T_{k+1} = T_{k} + \left(-\dfrac{T_{k}-T_{ambient}}{\tau} + u + d \right)\Delta t
$$

Then we can visualise the temperature over time:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


dt = 0.01 # Time step
time = 10 # Total simulation time
t = np.arange(0, time, dt)

# System state
temp = np.zeros_like(t)   # temperature
u = np.zeros_like(t)      # control input

# Initial conditions
temp[0] = 30

# Desired temperature
target = 25

# Bang-bang parameters
u_max = 5.0  # Maximum control input (e.g., heater power)

# Simulation
for k in range(len(t)-1):

    error = target - temp[k]

    # Bang-bang controller
    if error < 0:
        u[k] = -u_max
    else:
        u[k] = 0


    # Random process disturbance + Ambient heat exchange
    disturbance = np.random.normal(0, 1)


    # Dynamics
    tau = 1.0  # Thermal time constant
    ambient_temp = 30  # Ambient temperature

    temp[k+1] = temp[k] + (-(temp[k] - ambient_temp) / tau + u[k] + disturbance) * dt


# Plotting

fig, ax = plt.subplots(1, 1, figsize=(14, 6), sharex=True)

ax.plot(t, temp, label="Temperature")
ax.axhline(target, color='r', linestyle='--', label='Setpoint')
ax.set_ylabel("Temperature (°C)")
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.show()

Messing around with the values and you'll see something fun: If you let $u_max$ be too small, the system never reaches the desired state. Specifically, if 
$$
u_{max} < \left(-\dfrac{T + T_{ambient}}{\tau} + d\right)_{max}
$$
then the system **doesn't have enough control authority**. Simply, it means that the actuator isn't strong enough to combat the disturbances. In those cases, the controller will saturate, and the system maintains relative balance around the steady state error 
$$
-\dfrac{T + T_{ambient}}{\tau} + u_{max} + d
$$
Looking closely around the setpoint, you'd also see a lot of switching on and off. This is called **chatterring**, and in some cases it can be a bad things. With IC switches and power electronics some controllers do rely on this constant switching, but if you use mechanical relays or simpler mechanical switches, it is generally better to remove it. 

We can do this by applying a deadband around the setpoint. This just means a band of values around $SP$ where the state is acceptable, and we don't need the control input. Also, we can use three states: $u_{max}$, 0, and $-u_{max}$. This would be equivalent to a room with an AC and a heater trying to keep the temperature around $24 - 26 \degree \text{C}$. Mathematically, we can write:

$$
u(t) = 
\begin{cases}
u_{max} & \text{if} \; T \geq SP + \delta  \\
0 & \text{if} \; SP - \delta < T < SP + \delta  \\
-u_{max} & \text{if} \; T \leq SP - \delta  \\
\end{cases}
$$